In [6]:
import os
import pandas as pd
import psycopg2
from psycopg2 import sql
from psycopg2.extensions import ISOLATION_LEVEL_AUTOCOMMIT
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL


# Configuration

load_dotenv()

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = int(os.getenv("DB_PORT", "5432"))
DB_NAME = os.getenv("DB_NAME")
CSV_PATH = r"C:\Users\user\Desktop\bank2\data\bank.csv"

if not all([DB_USER, DB_PASSWORD, DB_HOST, DB_PORT, DB_NAME]):
    raise ValueError("Variables .env manquantes")

if not os.path.exists(CSV_PATH):
    raise FileNotFoundError(f"Fichier introuvable: {os.path.abspath(CSV_PATH)}")


# Fonctions utilitaires

def first_not_null(series):
    s = series.dropna()
    return s.iloc[0] if not s.empty else None

def to_bool(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    value = str(value).strip().lower()
    return value in {"true", "1", "yes", "y", "t"}

# 1. Creer la base si elle n'existe pas

admin_conn = psycopg2.connect(
    dbname="postgres",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
admin_conn.set_isolation_level(ISOLATION_LEVEL_AUTOCOMMIT)

with admin_conn.cursor() as cur:
    cur.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB_NAME,))
    if cur.fetchone() is None:
        cur.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(DB_NAME)))

admin_conn.close()


# 2. Connexion SQLAlchemy

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(db_url)


# 3. Lecture et nettoyage du CSV

df = pd.read_csv(CSV_PATH)

print(f"Fichier: {CSV_PATH}")
print(f"Lignes: {len(df)}")
print(f"Colonnes: {list(df.columns)}")
print(f"segment_client null: {df['segment_client'].isna().sum() if 'segment_client' in df.columns else 'colonne absente'}")
print(f"produit null: {df['produit'].isna().sum() if 'produit' in df.columns else 'colonne absente'}")
print(f"agence null: {df['agence'].isna().sum() if 'agence' in df.columns else 'colonne absente'}")

required_columns = [
    "transaction_id", "client_id", "date_transaction", "montant", "devise",
    "taux_change_eur", "categorie", "produit", "agence", "type_operation",
    "statut", "score_credit_client", "segment_client", "is_anomaly",
    "categorie_risque"
]

missing_columns = [col for col in required_columns if col not in df.columns]
if missing_columns:
    raise ValueError(f"Colonnes manquantes dans le CSV: {missing_columns}")

text_cols = [
    "transaction_id", "client_id", "devise", "categorie", "produit",
    "agence", "type_operation", "statut", "segment_client", "categorie_risque"
]

for col in text_cols:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA, "<NA>": pd.NA})
    )

df["score_credit_client"] = pd.to_numeric(df["score_credit_client"], errors="coerce")
df["montant"] = pd.to_numeric(df["montant"], errors="coerce")
df["taux_change_eur"] = pd.to_numeric(df["taux_change_eur"], errors="coerce")
df["date_transaction"] = pd.to_datetime(df["date_transaction"], errors="coerce")
df["is_anomaly"] = df["is_anomaly"].apply(to_bool)


# 4. Recreer le schema

with engine.begin() as conn:
    conn.execute(text("DROP VIEW IF EXISTS vw_client_transactions CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS transactions CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS transaction CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS compte CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS client CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS agence CASCADE;"))
    conn.execute(text("DROP TABLE IF EXISTS produit CASCADE;"))

    conn.execute(text("""
        CREATE TABLE client (
            client_id VARCHAR(50) PRIMARY KEY,
            score_credit_client INTEGER NOT NULL CHECK (score_credit_client BETWEEN 0 AND 1000),
            segment_client VARCHAR(50) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE agence (
            agence_id VARCHAR(50) PRIMARY KEY,
            nom_agence VARCHAR(100) NOT NULL UNIQUE,
            ville VARCHAR(50) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE produit (
            produit_id VARCHAR(50) PRIMARY KEY,
            nom_produit VARCHAR(100) NOT NULL UNIQUE,
            categorie VARCHAR(50) NOT NULL
        );
    """))

    conn.execute(text("""
        CREATE TABLE compte (
            compte_id VARCHAR(100) PRIMARY KEY,
            client_id VARCHAR(50) NOT NULL REFERENCES client(client_id) ON DELETE CASCADE,
            produit_id VARCHAR(50) NOT NULL REFERENCES produit(produit_id) ON DELETE CASCADE,
            UNIQUE(client_id, produit_id)
        );
    """))

    conn.execute(text("""
        CREATE TABLE transactions (
            transaction_id VARCHAR(50) PRIMARY KEY,
            client_id VARCHAR(50) NOT NULL REFERENCES client(client_id) ON DELETE CASCADE,
            agence_id VARCHAR(50) REFERENCES agence(agence_id) ON DELETE SET NULL,
            produit_id VARCHAR(50) NOT NULL REFERENCES produit(produit_id) ON DELETE CASCADE,
            date_transaction TIMESTAMP NOT NULL,
            montant DECIMAL(15,4) NOT NULL CHECK (montant <> 0),
            devise VARCHAR(10) NOT NULL DEFAULT 'EUR',
            taux_change_eur DECIMAL(15,6) NOT NULL DEFAULT 1.0,
            type_operation VARCHAR(50) NOT NULL,
            statut VARCHAR(20) NOT NULL CHECK (statut IN ('Completed', 'Pending', 'Cancelled')),
            categorie_risque VARCHAR(20) CHECK (categorie_risque IN ('Low', 'Medium', 'High')),
            is_anomaly BOOLEAN NOT NULL DEFAULT FALSE
        );
    """))

print("Tables creees")


# 5. Construire et inserer les donnees


# ---------- CLIENT ----------
all_clients = (
    df[["client_id"]]
    .dropna(subset=["client_id"])
    .drop_duplicates()
    .copy()
)

client_info = (
    df.groupby("client_id", as_index=False)
      .agg({
          "score_credit_client": first_not_null,
          "segment_client": first_not_null
      })
)

df_client = all_clients.merge(client_info, on="client_id", how="left")

median_score = df["score_credit_client"].median(skipna=True)
if pd.isna(median_score):
    median_score = 500

df_client["score_credit_client"] = (
    pd.to_numeric(df_client["score_credit_client"], errors="coerce")
    .fillna(median_score)
    .clip(0, 1000)
    .round()
    .astype(int)
)

df_client["segment_client"] = df_client["segment_client"].fillna("Inconnu")
df_client["segment_client"] = df_client["segment_client"].replace("", "Inconnu")
df_client = df_client.drop_duplicates(subset=["client_id"])

df_client.to_sql("client", engine, if_exists="append", index=False, method="multi")
print(f"client: {len(df_client)} lignes")

with engine.connect() as conn:
    valid_client_ids = {
        row[0] for row in conn.execute(text("SELECT client_id FROM client"))
    }

# ---------- AGENCE ----------
df_agence = (
    df[["agence"]]
    .dropna(subset=["agence"])
    .drop_duplicates()
    .copy()
)

df_agence["agence_id"] = "A" + (
    df_agence["agence"].astype("category").cat.codes + 10
).astype(str)

df_agence = df_agence.rename(columns={"agence": "nom_agence"})
df_agence["ville"] = "France"
df_agence = df_agence[["agence_id", "nom_agence", "ville"]].drop_duplicates()

df_agence.to_sql("agence", engine, if_exists="append", index=False, method="multi")
print(f"agence: {len(df_agence)} lignes")

agence_map = dict(zip(df_agence["nom_agence"], df_agence["agence_id"]))

# ---------- PRODUIT ----------
df_produit = (
    df[["produit", "categorie"]]
    .dropna(subset=["produit"])
    .drop_duplicates(subset=["produit"])
    .copy()
)

df_produit = df_produit.rename(columns={"produit": "nom_produit"})
df_produit = df_produit.reset_index(drop=True)
df_produit["produit_id"] = "P" + (df_produit.index + 100).astype(str)
df_produit = df_produit[["produit_id", "nom_produit", "categorie"]].drop_duplicates()

df_produit.to_sql("produit", engine, if_exists="append", index=False, method="multi")
print(f"produit: {len(df_produit)} lignes")

produit_map = dict(zip(df_produit["nom_produit"], df_produit["produit_id"]))

# ---------- COMPTE ----------
df_compte = (
    df[["client_id", "produit"]]
    .dropna(subset=["client_id", "produit"])
    .drop_duplicates()
    .copy()
)

df_compte = df_compte[df_compte["client_id"].isin(valid_client_ids)].copy()
df_compte["produit_id"] = df_compte["produit"].map(produit_map)
df_compte = df_compte.dropna(subset=["produit_id"])

df_compte["compte_id"] = (
    df_compte["client_id"].astype(str) + "_" + df_compte["produit_id"].astype(str)
)

df_compte = df_compte[["compte_id", "client_id", "produit_id"]].drop_duplicates()

df_compte.to_sql("compte", engine, if_exists="append", index=False, method="multi")
print(f"compte: {len(df_compte)} lignes")

# ---------- TRANSACTIONS ----------
# ---------- TRANSACTIONS ----------
status_map = {
    "completed": "Completed",
    "complete": "Completed",
    "done": "Completed",
    "terminé": "Completed",
    "termine": "Completed",
    "effectué": "Completed",
    "effectue": "Completed",

    "pending": "Pending",
    "en attente": "Pending",
    "attente": "Pending",

    "cancelled": "Cancelled",
    "canceled": "Cancelled",
    "annulé": "Cancelled",
    "annule": "Cancelled"
}

risk_map = {
    "low": "Low",
    "faible": "Low",

    "medium": "Medium",
    "moyen": "Medium",
    "modere": "Medium",
    "modéré": "Medium",

    "high": "High",
    "eleve": "High",
    "élevé": "High"
}

df_transactions = df.copy()
df_transactions = df_transactions[df_transactions["client_id"].isin(valid_client_ids)].copy()

df_transactions["agence_id"] = df_transactions["agence"].map(agence_map)
df_transactions["produit_id"] = df_transactions["produit"].map(produit_map)

df_transactions["devise"] = df_transactions["devise"].fillna("EUR")
df_transactions["taux_change_eur"] = pd.to_numeric(df_transactions["taux_change_eur"], errors="coerce").fillna(1.0)
df_transactions["is_anomaly"] = df_transactions["is_anomaly"].fillna(False)

df_transactions["statut"] = (
    df_transactions["statut"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(status_map)
)

df_transactions["categorie_risque"] = (
    df_transactions["categorie_risque"]
    .astype("string")
    .str.strip()
    .str.lower()
    .map(risk_map)
)

df_transactions["date_transaction"] = pd.to_datetime(df_transactions["date_transaction"], errors="coerce")
df_transactions["montant"] = pd.to_numeric(df_transactions["montant"], errors="coerce")

print("Transactions avant filtre:", len(df_transactions))
print("statut null apres mapping:", df_transactions["statut"].isna().sum())
print("date null:", df_transactions["date_transaction"].isna().sum())
print("produit_id null:", df_transactions["produit_id"].isna().sum())
print("montant null:", df_transactions["montant"].isna().sum())
print("Valeurs statut originales:", df["statut"].dropna().astype(str).str.strip().unique())

df_transactions = df_transactions.dropna(
    subset=[
        "transaction_id", "client_id", "produit_id", "date_transaction",
        "montant", "devise", "taux_change_eur", "type_operation", "statut"
    ]
)

df_transactions = df_transactions[df_transactions["montant"] != 0].copy()

df_transactions = df_transactions[[
    "transaction_id", "client_id", "agence_id", "produit_id", "date_transaction",
    "montant", "devise", "taux_change_eur", "type_operation", "statut",
    "categorie_risque", "is_anomaly"
]].drop_duplicates(subset=["transaction_id"])

print("Transactions apres filtre:", len(df_transactions))

df_transactions.to_sql("transactions", engine, if_exists="append", index=False, method="multi")
print(f"transaction: {len(df_transactions)} lignes")


# 6. Indexes et vue

with engine.begin() as conn:
    conn.execute(text("CREATE INDEX idx_transactions_client ON transactions(client_id);"))
    conn.execute(text("CREATE INDEX idx_transactions_date ON transactions(date_transaction);"))
    conn.execute(text("CREATE INDEX idx_transactions_agence ON transactions(agence_id);"))
    conn.execute(text("CREATE INDEX idx_client_segment ON client(segment_client);"))

    conn.execute(text("""
        CREATE VIEW vw_client_transactions AS
        SELECT
            c.client_id,
            c.segment_client,
            c.score_credit_client,
            t.transaction_id,
            t.date_transaction,
            t.montant,
            t.statut,
            a.nom_agence,
            a.ville,
            p.nom_produit,
            p.categorie
        FROM client c
        JOIN transactions t ON c.client_id = t.client_id
        LEFT JOIN agence a ON t.agence_id = a.agence_id
        JOIN produit p ON t.produit_id = p.produit_id;
    """))

print("Indexes crees")


# 7. Validation

with engine.connect() as conn:
    tables = ["client", "agence", "produit", "compte", "transactions"]
    for table in tables:
        count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
        print(f"{table}: {count} enregistrements")


# 8. Requetes analytiques

analytical_queries = {
    "total_transactions_by_agence_month": """
        SELECT
            a.nom_agence,
            DATE_TRUNC('month', t.date_transaction) AS mois,
            COUNT(t.transaction_id) AS nombre_transactions,
            SUM(t.montant) AS total_montant,
            AVG(t.montant) AS moyenne_montant
        FROM transactions t
        LEFT JOIN agence a ON t.agence_id = a.agence_id
        GROUP BY a.nom_agence, mois
        ORDER BY mois DESC, total_montant DESC;
    """,

    "clients_below_average_balance": """
        WITH avg_balance AS (
            SELECT AVG(total_balance) AS moyenne_nationale
            FROM (
                SELECT client_id, SUM(montant) AS total_balance
                FROM transactions
                WHERE statut = 'Completed'
                GROUP BY client_id
            ) AS client_balances
        )
        SELECT
            c.client_id,
            c.segment_client,
            SUM(t.montant) AS solde_client
        FROM client c
        JOIN transactions t ON c.client_id = t.client_id
        WHERE t.statut = 'Completed'
        GROUP BY c.client_id, c.segment_client
        HAVING SUM(t.montant) < (SELECT moyenne_nationale FROM avg_balance);
    """,

    "default_rate_by_risk_segment": """
        SELECT
            c.segment_client,
            t.categorie_risque,
            COUNT(t.transaction_id) AS total_transactions,
            SUM(CASE WHEN t.statut = 'Cancelled' THEN 1 ELSE 0 END) AS nombre_defauts,
            ROUND(
                CAST(
                    SUM(CASE WHEN t.statut = 'Cancelled' THEN 1 ELSE 0 END) * 100.0
                    / NULLIF(COUNT(t.transaction_id), 0)
                AS NUMERIC),
                2
            ) AS taux_defaut
        FROM client c
        JOIN transactions t ON c.client_id = t.client_id
        GROUP BY c.segment_client, t.categorie_risque
        ORDER BY taux_defaut DESC NULLS LAST;
    """,

    "top_10_agencies": """
        SELECT
            a.nom_agence,
            COUNT(t.transaction_id) AS nombre_transactions,
            SUM(t.montant) AS total_montant
        FROM agence a
        JOIN transactions t ON a.agence_id = t.agence_id
        WHERE t.statut = 'Completed'
        GROUP BY a.nom_agence
        ORDER BY total_montant DESC
        LIMIT 10;
    """
}

for query_name, query in analytical_queries.items():
    result = pd.read_sql(query, engine)
    print(f"\n{query_name}:")
    print(result.head())
    result.to_csv(f"{query_name}.csv", index=False)
    
print("\nPipeline termine")

Fichier: C:\Users\user\Desktop\bank2\data\bank.csv
Lignes: 2000
Colonnes: ['transaction_id', 'client_id', 'date_transaction', 'montant', 'devise', 'taux_change_eur', 'montant_eur', 'categorie', 'produit', 'agence', 'type_operation', 'statut', 'score_credit_client', 'segment_client', 'solde_avant', 'taux_interet', 'is_anomaly', 'annees', 'mois', 'jours_semaine', 'trimestre', 'montant_eur_ver', 'diff', 'categorie_risque']
segment_client null: 100
produit null: 0
agence null: 60
Tables creees
client: 150 lignes
agence: 8 lignes
produit: 8 lignes
compte: 975 lignes
Transactions avant filtre: 2000
statut null apres mapping: 113
date null: 75
produit_id null: 0
montant null: 0
Valeurs statut originales: ['Complete' 'Rejete' 'En attente']
Transactions apres filtre: 1804
transaction: 1804 lignes
Indexes crees
client: 150 enregistrements
agence: 8 enregistrements
produit: 8 enregistrements
compte: 975 enregistrements
transactions: 1804 enregistrements

total_transactions_by_agence_month:
      

In [7]:
import pandas as pd
df = pd.read_csv(r"C:\Users\user\Desktop\bank2\data\bank.csv")
print(df.columns.tolist())
print(df.head())

['transaction_id', 'client_id', 'date_transaction', 'montant', 'devise', 'taux_change_eur', 'montant_eur', 'categorie', 'produit', 'agence', 'type_operation', 'statut', 'score_credit_client', 'segment_client', 'solde_avant', 'taux_interet', 'is_anomaly', 'annees', 'mois', 'jours_semaine', 'trimestre', 'montant_eur_ver', 'diff', 'categorie_risque']
  transaction_id client_id     date_transaction  montant devise  \
0      TXN000559   CLI0060  2022-04-19 02:31:00  2050.42    EUR   
1      TXN001154   CLI0057  2024-06-20 20:51:00  -123.66    GBP   
2      TXN000764   CLI0015  2024-08-28 05:03:00  -396.17    EUR   
3      TXN001598   CLI0045  2024-01-07 08:16:00   225.20    EUR   
4      TXN001873   CLI0034  2024-08-11 19:52:00   935.32    EUR   

   taux_change_eur  montant_eur      categorie              produit  \
0             1.00      2050.42  Depot especes       Compte Epargne   
1             0.86      -143.79    Retrait DAB  Credit Consommation   
2             1.00      -396.17   